# Raw Macro Data Cleaning and EDA

This notebook documents the Week 3 file-based ingestion cleaning logic for the manually collected X dataset. The goal is to make the cleaning decisions explicit before they are used in the production ETL runner.

## Cleaning Rules

- Load `raw_macro_data.csv` from `backend/data/`
- Inspect row counts, columns, and missing values
- Strip whitespace from all string columns
- Repair obvious mojibake in `text_content` when present
- Parse `date_published` using the current year assumption
- Fill any missing `date_published` values with the mode of the parsed dates
- Drop empty rows and duplicates before the production ETL step

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

DATA_PATH = Path("raw_macro_data.csv")
LAGOS_TZ = ZoneInfo("Africa/Lagos")


def repair_text(value: str) -> str:
    suspicious_tokens = ("â€", "â€™", "â€œ", "â€", "â€”")
    if any(token in value for token in suspicious_tokens):
        try:
            return value.encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return value
    return value


raw_df = pd.read_csv(DATA_PATH)
print("Rows:", len(raw_df))
print("Columns:", raw_df.columns.tolist())
print("Missing values:\n", raw_df.isna().sum())
raw_df.head()


In [ ]:
clean_df = raw_df.copy()
current_year = datetime.now(tz=LAGOS_TZ).year

for column in ["source", "topic_label", "text_content", "date_published", "reference_url"]:
    clean_df[column] = clean_df[column].fillna("").astype(str).str.strip()

clean_df["source"] = clean_df["source"].replace("", "x_manual").str.lower()
clean_df["topic_label"] = clean_df["topic_label"].replace("", "unlabeled")
clean_df["text_content"] = clean_df["text_content"].map(repair_text)
clean_df["reference_url"] = clean_df["reference_url"].replace("", pd.NA)

parsed_dates = pd.to_datetime(
    clean_df["date_published"].replace("", pd.NA).map(
        lambda value: f"{value} {current_year}" if pd.notna(value) else value
    ),
    format="%b %d %Y",
    errors="coerce",
)

default_timestamp = parsed_dates.dropna().mode().iloc[0]
parsed_dates = parsed_dates.fillna(default_timestamp)
clean_df["published_at"] = parsed_dates.map(
    lambda value: value.to_pydatetime().replace(tzinfo=LAGOS_TZ)
)

clean_df = clean_df.rename(
    columns={
        "text_content": "content",
        "reference_url": "source_url",
    }
)

clean_df = clean_df[["source", "topic_label", "content", "published_at", "source_url"]]
clean_df["content"] = clean_df["content"].str.strip()
clean_df = clean_df.loc[clean_df["content"] != ""]
clean_df = clean_df.drop_duplicates(subset=["source", "source_url", "content"], keep="first")

print("Cleaned rows:", len(clean_df))
clean_df.head()


## Production Handoff

The exact logic above is mirrored in `app/services/ingestion.py`, and the database insert step is handled by `app/etl/runner.py` so the notebook stays as the audit and exploration artifact while the production pipeline stays reusable.